###Lakeflow Declarative Pipelines aka Delta Live Tables

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, to_timestamp
from pyspark.sql.types import DoubleType


source_path = "/Volumes/workspace/bronze/bronzevolume/bookings/data/"
target_path = "/Volumes/workspace/silver/silvervolume/bookings/data/"
checkpoint_path = "/Volumes/workspace/silver/silvervolume/bookings/checkpoint/"


rules = "booking_id IS NOT NULL AND passenger_id IS NOT NULL"

In [0]:
df_bronze = spark.read.format("delta").load("/Volumes/workspace/bronze/bronzevolume/airports/data/")
df_bronze.display()


In [0]:
df_transformed = df_bronze\
                .withColumn("amount", col("amount").cast(DoubleType()))\
                .withColumn("modifiedDate", current_timestamp())\
                .withColumn("booking_date", to_date(col("booking_date")))\
                .drop("_rescued_data")

In [0]:
df_cleaned = df_transformed.filter(rules)

query = (df_cleaned.writeStream.format("delta")\
                .outputMode("append")\
                .trigger(once=True)
                .option("checkpointLocation", checkpoint_path)\
                .start(target_path))